# IMAS Reader Basics

This notebook is for a downstream user who only wants to read an exported SOLEDGE-HDG IMAS `.nc` file.

It assumes only:

- `imas-python`
- `numpy`
- `matplotlib`

The examples inspect the exported `summary`, `equilibrium`, and `plasma_profiles` IDSs and plot a few fields on the rectangular `(R, Z)` GGD mesh.

In [ ]:
import json

import imas
import matplotlib.pyplot as plt
import numpy as np

## Open one exported IMAS file

Replace the filename below with the `.nc` file you received.

In [ ]:
db_path = "imas_demo_full.nc"

In [ ]:
with imas.DBEntry(db_path, "r") as entry:
    summary = entry.get("summary", 0)
    equilibrium = entry.get("equilibrium", 0)
    plasma = entry.get("plasma_profiles", 0)

## Read compact summary metadata

Most run-level metadata is stored in `summary.code.parameters` as JSON.

In [ ]:
summary_params = json.loads(str(summary.code.parameters))
summary_params

In [ ]:
print("Description:", summary.description)
print("Workflow:", summary.simulation.workflow)
print("Puff rate:", summary_params.get("puff_rate"))
print("Recycling:", summary_params.get("recycling_coefficient"))

## Read equilibrium data on the GGD mesh

The current exporter stores equilibrium values on the `nodes` subset of a rectangular cylindrical GGD mesh.
The values are flattened in `(R, Z)` order and can be reshaped using the stored grid shape.

In [ ]:
eq_params = json.loads(str(equilibrium.code.parameters))
nr, nz = eq_params["grid_shape"]

eq_ggd = equilibrium.time_slice[0].ggd[0]
r = eq_ggd.r[0].values.reshape(nr, nz)
z = eq_ggd.z[0].values.reshape(nr, nz)
psi = eq_ggd.psi[0].values.reshape(nr, nz)
br = eq_ggd.b_field_r[0].values.reshape(nr, nz)
bz = eq_ggd.b_field_z[0].values.reshape(nr, nz)
bphi = eq_ggd.b_field_phi[0].values.reshape(nr, nz)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))
im = ax.pcolormesh(r, z, psi, shading="auto")
fig.colorbar(im, ax=ax, label="psi")
ax.set_xlabel("R [m]")
ax.set_ylabel("Z [m]")
ax.set_aspect("equal")
ax.set_title("Poloidal flux")
plt.show()

## Read plasma profiles on the same mesh

The first plasma export includes:

- electron density and temperature
- ion density, temperature, and parallel velocity
- neutral density

In [ ]:
plasma_ggd = plasma.ggd[0]

ne = plasma_ggd.electrons.density[0].values.reshape(nr, nz)
te = plasma_ggd.electrons.temperature[0].values.reshape(nr, nz)

ion = plasma_ggd.ion[0]
ni = ion.density[0].values.reshape(nr, nz)
ti = ion.temperature[0].values.reshape(nr, nz)
u_par = ion.velocity[0].parallel.reshape(nr, nz)

neutral = plasma_ggd.neutral[0]
nn = neutral.density[0].values.reshape(nr, nz)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

plots = [
    (ne, "Electron density"),
    (te, "Electron temperature"),
    (ti, "Ion temperature"),
    (nn, "Neutral density"),
]

for ax, (field, title) in zip(axes.flat, plots):
    im = ax.pcolormesh(r, z, field, shading="auto")
    fig.colorbar(im, ax=ax)
    ax.set_xlabel("R [m]")
    ax.set_ylabel("Z [m]")
    ax.set_aspect("equal")
    ax.set_title(title)

plt.show()

## A few practical notes

- Outside the original HDG mesh, the exporter writes `NaN`.
- The equilibrium and plasma values currently live on the `nodes` subset of the rectangular GGD mesh.
- The current exporter stores rectangular GGD values in flattened `(R, Z)` order, so the stored `grid_shape` should be used when reshaping.
- Some convention notes, such as the interpretation of `psi` and `j_phi`, are stored in `equilibrium.code.parameters`.